### Regularization

#### 0. Core idea

Add a penalty term to the loss based on model complexity (usually weight magnitude), so the optimizer has to trade off fitting the training data against keeping the model simple. Reduces variance (overfitting) at some cost to bias, the same bias/variance tradeoff already covered per-parameter in the XGBoost hyperparameters note, L1/L2 here are the same lambda mechanic applied to linear models instead of tree leaf weights.

Toy setup: two unregularized (plain OLS) weights from a linear model, before any penalty is applied: w_a = 0.3 (a weak feature), w_b = 0.8 (a strong feature). Penalty strength lambda = 0.5 throughout.


#### 1. L2 (Ridge): proportional shrinkage, never exactly zero

Penalty: lambda * sum(w^2). For a single weight, minimizing (w - w_ols)^2 + lambda*w^2 gives the closed form: w_ridge = w_ols / (1 + lambda).

Worked:
```
w_a: w_ridge = 0.3 / (1+0.5) = 0.3/1.5 = 0.200
w_b: w_ridge = 0.8 / (1+0.5) = 0.8/1.5 = 0.533
```
Both weights shrink by the same proportional factor (divide by 1.5), neither hits exactly zero, no matter how small the original weight or how large lambda gets (short of infinity). Ridge shrinks everything smoothly, it does not select features, it just tempers all of them.

#### 2. L1 (Lasso): constant pull, hits exactly zero

Penalty: lambda * sum(|w|). The closed-form update for a single weight (soft-thresholding): w_lasso = sign(w_ols) * max(|w_ols| - lambda, 0).

Worked, same lambda=0.5:
```
w_a: |0.3| - 0.5 = -0.2, max(-0.2, 0) = 0  ->  w_lasso = 0   (exactly zero)
w_b: |0.8| - 0.5 = 0.3,  max(0.3, 0) = 0.3 ->  w_lasso = 0.3
```
w_a, the weaker feature, gets pushed to EXACTLY zero, effectively removed from the model. w_b survives but still shrinks (0.8 to 0.3, more aggressively than Ridge's 0.8 to 0.533 at the same lambda). This is Lasso's defining property: it performs feature selection as a side effect of regularization, because the penalty's pull toward zero is a constant lambda regardless of how small the weight already is, once the weight is smaller than lambda, the optimal move is to zero it out entirely, not just shrink it a little.

#### 3. Elastic Net

Combines both penalties: lambda1*sum(|w|) + lambda2*sum(w^2). Gets some of Lasso's sparsity (weak features can still hit zero) plus some of Ridge's stability (correlated features get shrunk together rather than Lasso's tendency to arbitrarily keep one and zero out the other among a correlated group). A practical middle ground when you want feature selection but do not fully trust Lasso's behavior on correlated features.


In [ ]:
import numpy as np

w_ols = np.array([0.3, 0.8])
lam = 0.5

w_ridge = w_ols / (1 + lam)
w_lasso = np.sign(w_ols) * np.maximum(np.abs(w_ols) - lam, 0)

print("original weights:", w_ols)
print("Ridge weights:", w_ridge)
print("Lasso weights:", w_lasso, "(note the exact zero)")

#### 4. Dropout (neural networks)

Randomly zero out a fraction of neurons during training only, forces the network to not rely too heavily on any single neuron or co-adapted group of neurons, acts as an implicit ensemble of subnetworks, each training pass uses a different random subset of the network.

Worked example: a neuron with 100 inputs, each weight=1, each input=1, so neuron input = 100 with no dropout. At 40% dropout, roughly 40 of those 100 inputs get zeroed out during training, effective input drops to about 60. But at inference time, dropout is OFF, all 100 inputs contribute, giving 100, a train/inference mismatch in scale that needs correcting.

Fix, inverted dropout: sample a mask where each unit is kept with probability (1-p), zeroed with probability p, apply it, then rescale the kept units by 1/(1-p) to compensate.
```
p = 0.4 (40% dropout)
kept sum during training ~ 60
rescaled: 60 * (1/(1-0.4)) = 60 * 1.667 = 100
```
Training and inference now see the same expected scale, no correction needed at inference time, the model at eval time just uses every neuron with no zeroing and no rescaling. Common rates: 0.1-0.3 for large models, 0.4-0.5 for smaller fully-connected nets, too high a rate underfits, the network loses too much capacity per step to learn anything.

#### 5. Early stopping

Stop training once validation loss stops improving, even if training loss keeps dropping. Not a penalty term added to the loss, a different mechanism entirely, it regularizes by simply not letting the model train long enough to fully memorize the training set. Directly relevant to every boosting model in this series (n_estimators is exactly the thing early stopping caps).

#### 6. Tree-specific regularization (max_depth, min_samples_leaf, gamma, min_child_weight)

Already covered in depth in `boosting.ipynb`'s hyperparameters section and `bagging.ipynb`'s Random Forest notes, the same complexity-penalty idea, expressed through tree structure limits instead of a weight-penalty term in a loss function.

In [ ]:
import torch
import torch.nn as nn

dropout_layer = nn.Dropout(0.2)
p = 0.2
x = torch.randn((1, 5))

print("input:", x)
print("manual inverted-dropout scaling (x / (1-p)):", x / (1 - p))
print("nn.Dropout output (training mode):", dropout_layer(x))

dropout_layer.eval()
print("nn.Dropout output (eval mode, no drop, no scaling):", dropout_layer(x))

In [ ]:
from sklearn.linear_model import Ridge, Lasso, LinearRegression
import numpy as np

rng = np.random.default_rng(0)
X = rng.normal(size=(50, 10))
true_w = np.array([2, -1.5, 0, 0, 0, 0, 0, 0, 0, 0])  # only first 2 features matter
y = X @ true_w + rng.normal(scale=0.5, size=50)

ols = LinearRegression().fit(X, y)
ridge = Ridge(alpha=1.0).fit(X, y)
lasso = Lasso(alpha=0.1).fit(X, y)

print("OLS coefficients:  ", ols.coef_.round(2))
print("Ridge coefficients:", ridge.coef_.round(2))
print("Lasso coefficients:", lasso.coef_.round(2))
print("\nLasso zeroed-out features:", np.sum(np.isclose(lasso.coef_, 0)), "of 10 (true irrelevant count: 8)")